# 3-13 计数缺失值与排序

本节介绍类别统计、缺失值检查与处理，以及按数值排序。缺失值需要用专门的方法判断，不能简单写成 `== np.nan`。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备示例表格

In [ ]:
df = pd.DataFrame({
    "company": ["B", "A", "B", np.nan],
    "gender": ["female", "female", "male", "male"],
    "age": [40, 31, 28, 28]
})
df

## 2. value_counts：统计类别数量与占比

默认统计每个非缺失类别的数量，并按数量降序排列。`normalize=True` 返回占比。

In [ ]:
df["gender"].value_counts()

In [ ]:
df["gender"].value_counts(normalize=True)

男女各 2 人，占比都是 `0.5`。默认不计缺失值；要把缺失值也作为一类，使用 `dropna=False`。

In [ ]:
df["company"].value_counts(dropna=False)

## 3. isna 与 any：检查缺失值

`isna()` 逐个判断，缺失的位置为 `True`；`notna()` 判断非缺失。

In [ ]:
df.isna()

In [ ]:
df["company"].isna()

In [ ]:
df.loc[df["company"].isna()]

### 3.1 判断哪些列存在缺失

`any()` 默认沿行方向汇总，也就是对每一列检查“是否至少存在一个 True”。

In [ ]:
df.isna().any()

只有 `company` 为 `True`。布尔值求和可以统计每列的缺失数量。

In [ ]:
df.isna().sum()

若检查每一行是否存在缺失，设置 `axis=1`。

In [ ]:
df.isna().any(axis=1)

## 4. dropna：删除缺失数据

默认删除至少有一个缺失值的行；`subset` 指定只检查哪些列。方法默认返回新对象。

In [ ]:
df.dropna()

In [ ]:
df.dropna(subset=["company"])

本例都会去掉最后一行。如果只希望删除所有字段都为空的行，可以指定 `how="all"`。

## 5. fillna、ffill 与 bfill：填充缺失数据

| 方法 | 作用 |
|---|---|
| `fillna(值或字典)` | 用指定内容填充 |
| `ffill()` | 用前一个有效值填充 |
| `bfill()` | 用后一个有效值填充 |

### 5.1 使用固定值

字典可以为不同列指定不同填充值。

In [ ]:
df.fillna({"company": "未知"})

### 5.2 前向填充与后向填充

前向填充沿当前行顺序，使用前一个有效值；后向填充使用后一个有效值。只有顺序有实际意义时，才适合这种方法。

In [ ]:
df.ffill()

In [ ]:
df.bfill()

`ffill()` 把最后一行公司填成 B；`bfill()` 无法填充最后一行，因为它后面没有可用值。向前填充使用 `ffill()`，向后填充使用 `bfill()`。

## 6. sort_values：按数据排序

`by` 指定排序字段，`ascending=False` 表示降序。

In [ ]:
df.sort_values(by="age", ascending=False)

多个字段可分别设置方向。`na_position` 控制缺失值放在前面或后面，`ignore_index=True` 让结果重新编号。

In [ ]:
df.sort_values(by=["age", "company"], ascending=[False, True], na_position="last", ignore_index=True)

排序与填充都会默认返回新表；原始 `df` 仍然保留最初的内容。

## 7. 要点总结

- `value_counts` 统计类别，`normalize=True` 统计占比，`dropna=False` 纳入缺失类别。
- `isna()` 判断缺失，配合 `any()` 看是否存在，配合 `sum()` 看数量。
- `dropna` 删除缺失记录，`fillna` / `ffill` / `bfill` 按指定规则填充。
- 前后填充依赖行顺序；按值排序使用 `sort_values`。